# EFUS 2017 — presentation pipeline (London, August)

End-to-end pipeline that reproduces every figure in the 5-slide talk, ending in the
recap triptych. Arm: **London (GOR 7), August, living room, 26 °C fixed criterion**.

Stages:
1. **Data** — EFUS hourly parquet + the daily-overheating dataset.
2. **Fig 1 (motivation)** — whole-stock overheating vs outdoor 2DMMT, mean + 90% CI.
3. **Fig 2 (model)** — per-dwelling indoor–outdoor lines motivating random intercept/slope.
4. **Fig 3 (M7 forest)** — static LME M7 fixed effects (all + significant); fabric null.
5. **Fig 4 (variance)** — M1–M7 variance accounted for vs the null (~half).
6. **Fig 5 (projection)** — UKCP18 12 km RCM RCP8.5: C+ vs rest to 2080.
7. **Triptych** — (a) dose-response, (b) M7 forest, (c) projection.

Upstream artefacts consumed (regenerate commands in the relevant sections):
- `analysis/livingroom/august/daily_overheating.parquet` ← `python/efus_overheating_datasets.py --room livingroom --region london --subdir august --months 8`
- `diagnostics/london_august_varcomp.csv` ← `Rscript R/efus_mm_london_1month.R`
- `analysis/livingroom/august/climate_projection.csv` ← `python/efus_climate_projection.py --room livingroom --region london --subdir august`

M7 itself is **refit live below** via `nlme` (fast static fit).

## 0. Setup

In [ ]:
import os, subprocess, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import scienceplots
    plt.style.use(["science", "nature", "bright"])
except Exception:
    pass
%matplotlib inline

ROOT = os.getcwd()
THR  = 26                      # fixed living-room threshold (deg C)
OUTDIR = "plots/efus2017/london_livingroom_1month"
os.makedirs(f"{OUTDIR}/description", exist_ok=True)
os.makedirs(f"{OUTDIR}/lme", exist_ok=True)
os.makedirs(f"{OUTDIR}/impact_profiles", exist_ok=True)
print("ROOT:", ROOT)

## 1. Data

`raw_aug` — hourly London-August living-room observations (for the random-effects
illustration). `daily` — one row per dwelling-day with the overheating flag, 2DMMT,
and building characteristics (built by `efus_overheating_datasets.py`).

In [ ]:
BLDG_CSV = "test_data/ukda_9434_csv_r/csv/selected_interview_responses_caseid.csv"
bldg = pd.read_csv(BLDG_CSV, usecols=["CaseID", "gorEHS_efus"]).drop_duplicates("CaseID")

raw = pd.read_parquet("efus_indoor_outdoor_livingroom.parquet",
                      columns=["CaseID", "hour", "T_in", "T_out"])
raw = raw[raw["T_in"].between(-10, 40) & raw["T_out"].between(-10, 40)]
raw = raw.merge(bldg, on="CaseID", how="left")
raw_aug = raw[(raw["gorEHS_efus"] == 7) & (raw["hour"].dt.month == 8)].copy()
raw_aug["T_out_c"] = raw_aug["T_out"] - raw_aug["T_out"].mean()

daily = pd.read_parquet("analysis/livingroom/august/daily_overheating.parquet")
EPC_W = daily.drop_duplicates("CaseID")["EPceeb12e_efus"].value_counts(normalize=True).to_dict()

print(f"raw_aug : {len(raw_aug):,} obs, {raw_aug.CaseID.nunique()} dwellings")
print(f"daily   : {len(daily):,} dwelling-days, {daily.CaseID.nunique()} dwellings")
print("EPC weights:", {int(k): round(v, 3) for k, v in sorted(EPC_W.items())})

## 2. Fig 1 (motivation) — overheating vs outdoor 2DMMT, whole stock

In [ ]:
def draw_dose_response(ax):
    g = daily.copy()
    g["bin"] = g["T_2DMMT"].round()
    a = g.groupby("bin")[f"daily_overheat_{THR}"].agg(["mean", "count"]).reset_index()
    a = a[a["count"] >= 5]
    p, n = a["mean"].to_numpy(), a["count"].to_numpy()
    se = np.sqrt(p * (1 - p) / n); z = 1.645
    lo, hi = np.clip(p - z*se, 0, 1), np.clip(p + z*se, 0, 1)
    x = a["bin"].to_numpy()
    ax.fill_between(x, 100*lo, 100*hi, color="#4477AA", alpha=0.25, label="90\\% CI")
    ax.plot(x, 100*p, color="#4477AA", lw=1.6, marker="o", ms=2.5, label="Mean")
    ax.set_xlabel("Outdoor 2-day mean max temperature (2DMMT) ($^\\circ$C)")
    ax.set_ylabel(r"Days exceeding overheating criterion (\%)")
    ax.set_ylim(0, 100); ax.set_xlim(x.min(), x.max())
    ax.grid(axis="y", alpha=0.4); ax.legend(fontsize=6, frameon=False, loc="upper left")

fig, ax = plt.subplots(figsize=(3.4, 2.6))
draw_dose_response(ax)
ax.set_title("Living-room overheating vs outdoor temperature\nEFUS 2017 London, August ($26^\\circ$C, $>3\\%$ occ. hrs)", fontsize=6)
fig.savefig(f"{OUTDIR}/description/fixed_exceedance_vs_2dmmt_stock.svg", bbox_inches="tight")
plt.show()

## 3. Fig 2 (model) — per-dwelling indoor–outdoor relationship

In [ ]:
def draw_random_effects(ax):
    rows = []
    for cid, sub in raw_aug.groupby("CaseID"):
        if len(sub) < 300: continue
        b, a = np.polyfit(sub["T_out_c"], sub["T_in"], 1)
        rows.append((cid, a, b))
    fit = pd.DataFrame(rows, columns=["CaseID", "intercept", "slope"]).sort_values("intercept")
    pick = fit.iloc[np.linspace(0, len(fit)-1, 6).round().astype(int)].reset_index(drop=True)
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    xr = np.linspace(raw_aug["T_out_c"].min(), raw_aug["T_out_c"].max(), 50)
    for i, r in pick.iterrows():
        sub = raw_aug[raw_aug["CaseID"] == r["CaseID"]]; c = colors[i % len(colors)]
        ax.scatter(sub["T_out_c"], sub["T_in"], s=2, alpha=0.10, color=c, edgecolors="none")
        ax.plot(xr, r["intercept"] + r["slope"]*xr, color=c, lw=1.4)
    B, A = np.polyfit(raw_aug["T_out_c"], raw_aug["T_in"], 1)
    ax.plot(xr, A + B*xr, color="k", lw=1.6, ls="--", label="Pooled fit")
    ax.set_xlabel("Centred outdoor temperature, $T_{\\mathrm{out,c}}$ ($^\\circ$C)")
    ax.set_ylabel("Indoor temperature, $T_{\\mathrm{in}}$ ($^\\circ$C)")
    ax.legend(fontsize=6, frameon=False, loc="upper left")

fig, ax = plt.subplots(figsize=(3.4, 2.6))
draw_random_effects(ax)
ax.set_title("Per-dwelling indoor--outdoor relationship\nEFUS 2017 London living rooms, August", fontsize=6)
fig.savefig(f"{OUTDIR}/description/lme_random_effects_illustration.svg", bbox_inches="tight")
plt.show()

## 4. Fit M7 (static LME) via nlme

`T_in ~ T_out_c + sin_h + cos_h + (CavityWall + InsulatedWalls + FullyDblGlz + AnyCooling)
+ factor(dwtype) + factor(dwage) + factor(floor6x) + factor(EPC)`, random `(T_out_c | dwelling)`.
Refit live and exported to a temp CSV.

In [ ]:
R_FIT = r'''
suppressMessages({library(nlme); library(arrow); library(lubridate)})
ROOT <- Sys.getenv("EFUS_ROOT")
df <- as.data.frame(arrow::read_parquet(file.path(ROOT,"efus_indoor_outdoor_livingroom.parquet")))
df <- df[df$T_in>=-10 & df$T_in<=40 & df$T_out>=-10 & df$T_out<=40, ]
B <- c("CaseID","dwtype_efus","dwage_efus","WallType2x_efus","InsulatedWalls_efus",
       "FullyDblGlz_efus","floor6x_efus","EPceeb12e_efus","gorEHS_efus","AnyCooling")
bldg <- read.csv(file.path(ROOT,"test_data/ukda_9434_csv_r/csv/selected_interview_responses_caseid.csv"),
                 stringsAsFactors=FALSE)[,B]
df <- merge(df,bldg,by="CaseID",all.x=TRUE); df <- df[complete.cases(df[,B[-1]]),]
for(c in B[-1]) df[[c]] <- as.integer(df[[c]])
df <- df[df$gorEHS_efus==7,]; df <- df[lubridate::month(df$hour)==8,]
df$T_out_c <- df$T_out-mean(df$T_out); h <- lubridate::hour(df$hour)
df$sin_h <- sin(2*pi*h/24); df$cos_h <- cos(2*pi*h/24)
n <- tapply(df$CaseID,df$CaseID,length); df <- df[df$CaseID %in% names(n)[n>=20],]
df$dwelling <- as.character(df$CaseID); df <- droplevels(df[order(df$dwelling,df$hour),])
df$CavityWall <- as.integer(df$WallType2x_efus==2)
ctrl <- lmeControl(opt="optim",maxIter=500,msMaxIter=500,tolerance=1e-6,niterEM=50)
BIN <- "CavityWall + InsulatedWalls_efus + FullyDblGlz_efus + AnyCooling"
CAT <- "factor(dwtype_efus)+factor(dwage_efus)+factor(floor6x_efus)+factor(EPceeb12e_efus)"
m7 <- lme(as.formula(paste("T_in ~ T_out_c + sin_h + cos_h +",BIN,"+",CAT)),
          data=df, random=~T_out_c|dwelling, method="ML", control=ctrl)
fe <- summary(m7)$tTable
out <- data.frame(term=rownames(fe), estimate=fe[,"Value"], se=fe[,"Std.Error"], p=fe[,"p-value"])
write.csv(out, Sys.getenv("EFUS_M7_OUT"), row.names=FALSE)
'''
m7_csv = os.path.join(tempfile.gettempdir(), "m7_aug_coefs.csv")
env = dict(os.environ, EFUS_ROOT=ROOT, EFUS_M7_OUT=m7_csv)
with tempfile.NamedTemporaryFile("w", suffix=".R", delete=False) as f:
    f.write(R_FIT); rpath = f.name
res = subprocess.run(["Rscript", rpath], env=env, capture_output=True, text=True)
if res.returncode != 0:
    print(res.stdout); print(res.stderr); raise RuntimeError("R fit failed")
m7 = pd.read_csv(m7_csv)
print(f"M7: {len(m7)} fixed-effect terms")
m7.head()

## 5. Fig 3 (M7 forest) — all + significant

In [ ]:
LAB = {
 "T_out_c":"Outdoor temp. (per $^\\circ$C)","sin_h":r"sin(hour)","cos_h":r"cos(hour)",
 "CavityWall":"Cavity wall","InsulatedWalls_efus":"Wall insulation",
 "FullyDblGlz_efus":"Full double glazing","AnyCooling":"Cooling",
 "factor(dwtype_efus)2":"Semi-detached","factor(dwtype_efus)3":"End-terrace",
 "factor(dwtype_efus)4":"Mid-terrace","factor(dwtype_efus)5":"Bungalow","factor(dwtype_efus)6":"Flat",
 "factor(dwage_efus)2":"Age 1919--44","factor(dwage_efus)3":"Age 1944--64",
 "factor(dwage_efus)4":"Age 1964--80","factor(dwage_efus)5":"Age 1980--90",
 "factor(dwage_efus)6":"Age 1990--2002","factor(dwage_efus)7":"Age post-2002",
 "factor(floor6x_efus)2":"Floor band 2","factor(floor6x_efus)3":"Floor band 3",
 "factor(floor6x_efus)4":"Floor band 4","factor(floor6x_efus)5":"Floor band 5","factor(floor6x_efus)6":"Floor band 6",
 "factor(EPceeb12e_efus)2":"EPC D (vs C+)","factor(EPceeb12e_efus)3":"EPC E (vs C+)","factor(EPceeb12e_efus)4":"EPC F/G (vs C+)",
}
def m7_table(sig_only=False):
    d = m7[m7["term"] != "(Intercept)"].copy()
    d["lo"] = d["estimate"] - 1.96*d["se"]; d["hi"] = d["estimate"] + 1.96*d["se"]
    d["sig"] = d["p"] < 0.05
    d["label"] = d["term"].map(LAB).fillna(d["term"])
    if sig_only: d = d[d["sig"]]
    return d.iloc[::-1].reset_index(drop=True)

def draw_forest(ax, sig_only=False):
    d = m7_table(sig_only); y = np.arange(len(d))
    for yi, (_, r) in zip(y, d.iterrows()):
        c = "#CC3311" if r["sig"] else "#999999"
        ax.plot([r["lo"], r["hi"]], [yi, yi], color=c, lw=1.2)
        ax.scatter([r["estimate"]], [yi], s=13, color=c, edgecolors="none")
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_yticks(y); ax.set_yticklabels(d["label"], fontsize=5)
    ax.set_xlabel("Effect on indoor temperature ($^\\circ$C, 95\\% CI)", fontsize=6)
    ax.grid(axis="x", alpha=0.3)

for sig, fname, ttl in [(False,"m7_forest_all","M7 fixed effects"),
                        (True,"m7_forest_significant","M7 significant effects ($p<0.05$)")]:
    d = m7_table(sig)
    fig, ax = plt.subplots(figsize=(3.4, max(2.2, 0.22*len(d)+0.6)))
    draw_forest(ax, sig)
    ax.set_title(f"{ttl} --- London, August", fontsize=6)
    fig.savefig(f"{OUTDIR}/lme/{fname}.svg", bbox_inches="tight")
    plt.show()

## 6. Fig 4 (variance) — M1–M7 variance accounted for vs null

Reads `diagnostics/london_august_varcomp.csv` (`Rscript R/efus_mm_london_1month.R`).
`% = 100 (1 - V_model / V_null)` with the static-family null `M0_static`.

In [ ]:
vc = pd.read_csv("diagnostics/london_august_varcomp.csv").set_index("model")
V0 = vc.loc["M0_static", "total_obs_var"]
models = ["M1","M2","M3","M4","M5","M6","M7"]
TOD = {"M1":"$T_{out}$","M2":"+sin/cos","M3":"+hour","M4":"+bin",
       "M5":"+bin+cat","M6":"sin/cos+bin","M7":"sin/cos+bin+cat"}
pvar = [100*(1 - vc.loc[m,"total_obs_var"]/V0) for m in models]

fig, ax = plt.subplots(figsize=(3.6, 2.6))
x = np.arange(len(models))
ax.bar(x, pvar, color="#4477AA", edgecolor="black", linewidth=0.4, width=0.78)
ax.axhline(50, color="#CC3311", ls="--", lw=1.0)
ax.text(len(models)-0.5, 51, "50\\% of total variance", color="#CC3311", fontsize=5.5, ha="right", va="bottom")
for xi, p in zip(x, pvar): ax.text(xi, p+1, f"{p:.0f}", ha="center", va="bottom", fontsize=5.5)
ax.set_xticks(x); ax.set_xticklabels([f"{m}\n{TOD[m]}" for m in models], fontsize=5)
ax.set_ylabel(r"Variance accounted for vs.\ null (\%)", fontsize=6.5)
ax.set_ylim(0, 60); ax.set_title("Static LME (M1--M7): variance explained\nLondon living rooms, August", fontsize=6)
fig.savefig(f"{OUTDIR}/lme/m1_m7_variance_explained.svg", bbox_inches="tight")
plt.show()
print({m: round(p,1) for m,p in zip(models, pvar)})

## 7. Fig 5 (projection) — C+ vs rest to 2080

Reads `analysis/livingroom/august/climate_projection.csv`
(`efus_climate_projection.py`, UKCP18 12 km RCM RCP8.5).

In [ ]:
proj = pd.read_csv("analysis/livingroom/august/climate_projection.csv")
proj = proj[proj["threshold"] == THR]

def proj_series(epcs):
    s = proj[proj["epc"].isin(epcs)].copy()
    w = {e: EPC_W[e] for e in epcs}; tot = sum(w.values())
    s["w"] = s["epc"].map(lambda e: w[e]/tot)
    a = (s.groupby(["member","year"])
           .apply(lambda g: np.average(g["mean_exceed_pct"], weights=g["w"]))
           .rename("v").reset_index())
    piv = a.pivot(index="year", columns="member", values="v").sort_index()
    piv = piv.rolling(6, center=True, min_periods=3).mean()
    return piv.index.values, piv.mean(1).values, piv.quantile(.1,1).values, piv.quantile(.9,1).values

GROUPS = [([1], "EPC C+ (most efficient)", "#CC3311"),
          ([2,3,4], "EPC D--F/G (rest)", "#4477AA")]

def draw_projection(ax):
    for epcs, lab, col in GROUPS:
        yr, m, lo, hi = proj_series(epcs)
        ax.fill_between(yr, lo, hi, color=col, alpha=0.18)
        ax.plot(yr, m, color=col, lw=1.7, label=lab)
    ax.axvline(2020, color="grey", lw=0.7, ls=":")
    ax.set_xlim(1985, 2078); ax.set_ylim(0, None)
    ax.set_xlabel("Year"); ax.set_ylabel(r"Days exceeding criterion (\%)")
    ax.legend(fontsize=5.5, frameon=False, loc="upper left")

fig, ax = plt.subplots(figsize=(3.6, 2.6))
draw_projection(ax)
ax.set_title("Projected overheating by EPC: C+ vs the rest\nLondon August, $26^\\circ$C, UKCP18 12\\,km RCM, RCP8.5", fontsize=6)
fig.savefig(f"{OUTDIR}/impact_profiles/climate_projection_slide.svg", bbox_inches="tight")
plt.show()

## 8. Recap triptych — (a) dose-response, (b) M7 forest, (c) projection

In [ ]:
fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(7.4, 2.4))

draw_dose_response(axA)
axA.get_legend().remove()
axA.set_xlabel("Outdoor 2DMMT ($^\\circ$C)", fontsize=6)
axA.set_ylabel("Days over criterion (\%)", fontsize=6)
axA.set_title("(a) Overheating rises with heat", fontsize=6.5)

draw_forest(axB, sig_only=True)
axB.set_xlabel("Effect on $T_{in}$ ($^\\circ$C)", fontsize=6)
axB.set_title("(b) Fabric not significant", fontsize=6.5)
axB.text(0.97, 0.05, "insulation,\ncavity, glazing\n= n.s.", transform=axB.transAxes,
         fontsize=4.6, ha="right", va="bottom", color="#555555", style="italic")

draw_projection(axC)
axC.set_xlabel("Year", fontsize=6); axC.set_ylabel("Days over criterion (\%)", fontsize=6)
axC.set_title("(c) Worsens to 2080 (C+ worst)", fontsize=6.5)

for ax in (axA, axB, axC): ax.tick_params(labelsize=5.5)
plt.tight_layout(w_pad=1.2)
fig.savefig(f"{OUTDIR}/impact_profiles/recap_triptych.svg", bbox_inches="tight")
plt.show()